# Extract frozen GLIM text tokens (+ pooled vectors) -- Gate 2

Route B, Gate 2. **GPU.** Attach the canonical sharded dataset
`thestonedape/task-aware-eegtotext` (v1) and the checkpoint
`thestonedape/glim-zuco-checkpoint`; enable Internet and the secret `GITHUB_TOKEN`.

**One complete run.** For every development (train+val) text identity it extracts
the unpooled `encode_text` hidden states `[L,1024]` + attention mask (the MaxSim
text tokens) and co-stores the pooled `embed_text` vector `[1024]` from the same
pass. Dedup + trial->text mapping match the frozen text-*vector* pipeline, so the
candidate set is identical. Held-out test is never embedded.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '0be3e36c410f4eda4c9137b928032ad5b4519bd4'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = 'e1f202cb793cfe7292fbc0072a4c26a7dd0660d9'
GLIM_WORKTREE = '/kaggle/working/GLIM'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
OUTPUT = '/kaggle/working/task-aware-eeg2text-glim-text-tokens'
CHUNK_SIZE = 256
DTYPE = 'float16'
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(len(v) == 64 for v in (EXPECTED_INDEX_SHA256, CHECKPOINT_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.0'], check=True)
env = os.environ.copy(); env['PYTHONPATH'] = WORKTREE
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'evaluation.test_extract_frozen_glim_text_tokens'], check=True, cwd=WORKTREE, env=env)
print({'clone': 'PASS', 'text_token_tests': 'PASS'})

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, ('Attach exactly one canonical sharded dataset', manifest_paths)
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, ('Attach exactly one GLIM checkpoint', checkpoint_paths)
checkpoint = checkpoint_paths[0]
state = hashlib.sha256()
with open(checkpoint, 'rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        state.update(block)
assert state.hexdigest() == CHECKPOINT_SHA256, 'checkpoint SHA mismatch'
print({'dataset_root': dataset_root, 'checkpoint': checkpoint})

In [ ]:
# ONE complete run over all development text identities. Resumable -- hash-valid
# chunks are reused, so re-run this cell after any interruption.
cmd = [
    sys.executable, '-B', os.path.join(WORKTREE, 'evaluation', 'extract_frozen_glim_text_tokens.py'),
    '--dataset-root', dataset_root, '--output-root', OUTPUT, '--glim-root', GLIM_WORKTREE,
    '--checkpoint', checkpoint, '--glim-commit', GLIM_COMMIT,
    '--expected-index-sha256', EXPECTED_INDEX_SHA256,
    '--expected-checkpoint-sha256', CHECKPOINT_SHA256,
    '--device', 'cuda', '--chunk-size', str(CHUNK_SIZE), '--dtype', DTYPE,
]
subprocess.run(cmd, check=True, cwd=WORKTREE, env={**os.environ, 'PYTHONPATH': WORKTREE})

In [ ]:
import numpy as np
sys.path.insert(0, WORKTREE)
from evaluation.extract_frozen_glim_text_tokens import text_token_chunk_sha256
man = json.load(open(os.path.join(OUTPUT, 'text_token_manifest.json'), encoding='utf-8'))
assert man['status'] == 'pass' and man['glim_commit'] == GLIM_COMMIT and man['dtype'] == DTYPE
assert man['token_dim'] == 1024 and man['checks']['held_out_test_accessed'] is False
assert 'test' not in man['split_counts'], man['split_counts']
# Integrity: recompute the array-bytes hash of each chunk (tokens+masks+vectors+ids).
for entry in man['chunks']:
    npz = os.path.join(OUTPUT, entry['token_file'])
    meta = json.load(open(npz[:-4] + '.json', encoding='utf-8'))
    with np.load(npz) as arch:
        assert set(arch.files) == {'tokens', 'masks', 'vectors'}, arch.files
        recomputed = text_token_chunk_sha256(arch['tokens'], arch['masks'], arch['vectors'], meta['text_target_ids'])
    assert recomputed == entry['sha256'] == meta['sha256'], entry['token_file']
run_metadata = {
    'status': 'pass', 'project_commit': COMMIT, 'glim_commit': GLIM_COMMIT,
    'checkpoint_sha256': CHECKPOINT_SHA256, 'dataset_index_sha256': EXPECTED_INDEX_SHA256,
    'unique_text_identities': man['unique_text_identities'], 'mapped_trials': man['mapped_trials'],
    'token_len': man['token_len'], 'token_dim': man['token_dim'], 'dtype': DTYPE,
    'combined_chunk_sha256': man['combined_chunk_sha256'], 'text_model_id': man['text_model_id'],
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True); handle.write('\n')
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
print({'identities': man['unique_text_identities'], 'mapped_trials': man['mapped_trials'],
       'token_len': man['token_len'], 'chunks': man['num_chunks'],
       'combined_chunk_sha256': man['combined_chunk_sha256']})
print('GLIM TEXT TOKEN EXTRACTION: PASS')

After PASS, save the printed OUTPUT (`task-aware-eeg2text-glim-text-tokens`) as a
new **private** Kaggle dataset (Version 1): text tokens `[L,1024]` + masks +
pooled vectors, the index, the trial->text mapping, and the manifest.

Next (Gate 2, `gate2_pooled_vector_identity`): confirm the co-stored pooled
vectors reproduce the frozen pooled vectors (EEG token run vs P4b EEG vectors;
this text run vs the frozen text vectors), then freeze the pooled-vs-token design.
Held-out test stays sealed.